# VantageGro Retail Stock Insight
## Stage 5: Machine Learning Model Training & Comparative Evaluation

### Objectives
1. **Target Specification**: Predict true unconstrained customer `Demand` rather than inventory-constrained `Units Sold`.
2. **Strict Chronological Splitting**: Train (70%), Validation (15%), Test (15%) preserving temporal order.
3. **Baselines**: Evaluate Naive Lag 1 demand and 7-day Moving Average.
4. **Candidate Regressors**: Train Ridge, Random Forest, HistGradientBoosting, and XGBoost.
5. **Comparative Metrics**: Calculate actual MAE, RMSE, R², and WAPE %.
6. **Model Artifact Persistence**: Save champion model to `models/vantagegro_best_model.joblib`.
7. **Diagnostic Visualizations**: Actual vs Predicted, Residuals, Feature Importance, and Category Errors.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (10, 5)
print('Model Training environment ready!')

### 1. Load Engineered Feature Matrix

In [ ]:
feat_path = 'data/processed/vantagegro_features.csv'
if not os.path.exists(feat_path):
    feat_path = '../data/processed/vantagegro_features.csv'

df = pd.read_csv(feat_path)
df['Date'] = pd.to_datetime(df['Date'])
print(f'Feature Matrix: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

### 2. Chronological Train / Validation / Test Splitting

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in numeric_cols if c not in ['Demand', 'Units Sold']]
target_col = 'Demand'

unique_dates = np.sort(df['Date'].unique())
n_dates = len(unique_dates)
train_end = unique_dates[int(n_dates * 0.70)]
val_end = unique_dates[int(n_dates * 0.85)]

train_df = df[df['Date'] <= train_end]
val_df = df[(df['Date'] > train_end) & (df['Date'] <= val_end)]
test_df = df[df['Date'] > val_end]

print(f'Train Slice:      {len(train_df):,} rows ({train_df["Date"].min().strftime("%Y-%m-%d")} to {train_df["Date"].max().strftime("%Y-%m-%d")})')
print(f'Validation Slice: {len(val_df):,} rows ({val_df["Date"].min().strftime("%Y-%m-%d")} to {val_df["Date"].max().strftime("%Y-%m-%d")})')
print(f'Test Slice:       {len(test_df):,} rows ({test_df["Date"].min().strftime("%Y-%m-%d")} to {test_df["Date"].max().strftime("%Y-%m-%d")})')

X_train, y_train = train_df[feature_cols].values, train_df[target_col].values
X_val, y_val = val_df[feature_cols].values, val_df[target_col].values
X_test, y_test = test_df[feature_cols].values, test_df[target_col].values

### 3. Model Training & Evaluation Function

In [ ]:
def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(np.mean((y_true - y_pred)**2))
    r2 = r2_score(y_true, y_pred)
    wape = np.sum(np.abs(y_true - y_pred)) / np.sum(y_true) * 100
    return {'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 'WAPE (%)': round(wape, 2), 'R2': round(r2, 4)}

results = {}
# Baseline 1: Naive Lag 1
results['Baseline: Naive Lag 1'] = evaluate(y_test, test_df['demand_lag_1'].values)
# Baseline 2: 7-Day Moving Average
results['Baseline: 7-Day MA'] = evaluate(y_test, test_df['rolling_mean_7'].values)

# Models
models = {
    'Ridge Regression': Ridge(alpha=1.0),
    'HistGradientBoosting': HistGradientBoostingRegressor(max_iter=150, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=80, max_depth=12, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=120, max_depth=6, learning_rate=0.08, random_state=42, n_jobs=-1)
}

trained = {}
test_preds = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    preds = m.predict(X_test)
    results[name] = evaluate(y_test, preds)
    trained[name] = m
    test_preds[name] = preds

res_df = pd.DataFrame(results).T
display(res_df)

### 4. Performance Visualizations

In [ ]:
# 1. Model WAPE Comparison
plt.figure(figsize=(10, 4))
colors = ['#94a3b8', '#94a3b8', '#3b82f6', '#10b981', '#3b82f6', '#3b82f6']
bars = plt.bar(res_df.index, res_df['WAPE (%)'], color=colors)
plt.title('Test Set Prediction Error (WAPE % - Lower is Better)', fontsize=12, fontweight='bold')
plt.ylabel('WAPE (%)')
plt.xticks(rotation=20, ha='right')
for b in bars:
    plt.text(b.get_x() + b.get_width()/2., b.get_height()+0.5, f"{b.get_height():.1f}%", ha='center', va='bottom', fontweight='bold', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# 2. Actual vs Predicted (Champion Model: HistGradientBoosting)
champion_preds = test_preds['HistGradientBoosting']
plt.figure(figsize=(7, 6))
plt.scatter(y_test[:1500], champion_preds[:1500], alpha=0.35, color='#10b981', s=15)
plt.plot([0, 400], [0, 400], color='black', linestyle='--')
plt.title('Actual vs Predicted Demand (HistGradientBoosting)', fontsize=11, fontweight='bold')
plt.xlabel('Actual Customer Demand')
plt.ylabel('Predicted Demand')
plt.tight_layout()
plt.show()